In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Đọc file dữ liệu (đảm bảo đường dẫn đúng với thư mục của bạn)
df = pd.read_csv('../data/raw/lung_cancer.csv')

# Hiển thị 5 dòng đầu tiên để kiểm tra
display(df.head())

# Kiểm tra tổng quan xem có giá trị null không
print(df.isnull().sum())
print(df.info())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Tách Features (X) và Target (y)
X = df.drop(columns=['lung_cancer_risk'])
y = df['lung_cancer_risk']

# Kiểm tra tỉ lệ mất cân bằng của class
print("Phân phối class ban đầu:\n", y.value_counts(normalize=True))

# Chia tập Train/Test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Các cột có giá trị liên tục cần được chuẩn hóa
continuous_cols = [
    'age', 'education_years', 'smoking_years', 'cigarettes_per_day', 
    'pack_years', 'air_pollution_index', 'bmi', 'oxygen_saturation', 
    'fev1_x10', 'crp_level', 'exercise_hours_per_week', 'alcohol_units_per_week'
]

scaler = StandardScaler()

# Chỉ fit trên tập Train để tránh rò rỉ dữ liệu (data leakage)
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Đã chuẩn hóa dữ liệu thành công!")

In [ ]:
from imblearn.over_sampling import SMOTE

# Áp dụng SMOTE chỉ trên tập Train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Phân phối trước SMOTE:\n", y_train.value_counts())
print("\nPhân phối sau SMOTE:\n", y_train_smote.value_counts())

In [ ]:
# Gộp lại và lưu thành file mới trong thư mục processed
processed_train = pd.concat([pd.DataFrame(X_train_smote, columns=X.columns), pd.Series(y_train_smote, name='lung_cancer_risk')], axis=1)
processed_test = pd.concat([pd.DataFrame(X_test, columns=X.columns), pd.Series(y_test, name='lung_cancer_risk')], axis=1)

processed_train.to_csv('../data/processed/train_processed.csv', index=False)
processed_test.to_csv('../data/processed/test_processed.csv', index=False)

print("Đã lưu dữ liệu tiền xử lý vào thư mục data/processed/")